# Simpsons Character Classification with Convolutional Neural Networks

---

In this project we build **image classifiers** that can recognize characters from *The Simpsons* TV series. We progress from a simple fully-connected baseline through a custom CNN to a modern **transfer-learning** approach (ResNet-18), all implemented in **PyTorch**.

<center><img src="https://i.imgur.com/i8zIGqX.jpg" style="text-align: center" height="300px"></center>

The dataset consists of frames extracted directly from episodes and was originally compiled by [Alexandre Attia](http://www.alexattia.fr/). It contains 18 labelled characters who may appear in various poses, positions, and even alongside other characters (though the target character is always the most prominent).

**Datasets:**

| Set | Link | Size |
|-----|------|------|
| Training | [Training data (OneDrive)](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219337&authkey=AMzI92bJPx8Sd60) | ~500 MB |
| External Test | [Test data (OneDrive)](https://onedrive.live.com/download?cid=C506CF0A4F373B0F&resid=C506CF0A4F373B0F%219341&authkey=ANnjK3Uq1FhuAe8) | ~10 MB |

We also download the same data from Kaggle for convenience (see below).

## 1. Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import shutil
import random
import glob
from pathlib import Path
from collections import Counter

# ── Scientific / data ────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# ── Image handling ───────────────────────────────────────────────────────────
import cv2
from PIL import Image

# ── PyTorch core ─────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# ── Torchvision ──────────────────────────────────────────────────────────────
import torchvision
from torchvision import datasets, transforms, models

# ── Utilities ────────────────────────────────────────────────────────────────
import kagglehub
from tqdm.auto import tqdm

# ── Configuration ────────────────────────────────────────────────────────────
IMG_SIZE = 128               # All images will be resized to IMG_SIZE x IMG_SIZE
BATCH_SIZE = 64
NUM_WORKERS = 4
SEED = 42
EPOCHS_FC = 20               # Fully-connected baseline
EPOCHS_CNN = 40              # Custom CNN
EPOCHS_TL_FROZEN = 10       # Transfer learning – feature extraction phase
EPOCHS_TL_FINETUNE = 15     # Transfer learning – fine-tuning phase
LR = 1e-3

# ── Reproducibility ─────────────────────────────────────────────────────────
def set_seed(seed: int = SEED):
    """Set random seeds for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Device ───────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

# Matplotlib defaults
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style("whitegrid")

## 2. Data Download & Preparation

We download the dataset from Kaggle: [The Simpsons Characters Dataset](https://www.kaggle.com/datasets/alexattia/the-simpsons-characters-dataset)

In [2]:
## Descargar dataset
path = kagglehub.dataset_download("alexattia/the-simpsons-characters-dataset")
print("Descargado en:", path)

## Definir rutas
source_path = os.path.join(path, "simpsons_dataset")
target_path = "./simpsons_dataset"

## Si ya existe, eliminar y copiar de nuevo
if os.path.exists(target_path):
    shutil.rmtree(target_path)
    print(f"⚠️ Carpeta existente eliminada: {target_path}")

shutil.copytree(source_path, target_path)
print(f"✅ Dataset copiado a: {target_path}")

Descargado en: /kaggle/input/the-simpsons-characters-dataset
⚠️ Carpeta existente eliminada: ./simpsons_dataset
✅ Dataset copiado a: ./simpsons_dataset


Podemos ver que los personajes están repartidos en carpetas

In [ ]:
## Los personajes se encuentran en carpetas
!ls $path/simpsons_dataset

abraham_grampa_simpson	  homer_simpson        otto_mann
agnes_skinner		  kent_brockman        patty_bouvier
apu_nahasapeemapetilon	  krusty_the_clown     principal_skinner
barney_gumble		  lenny_leonard        professor_john_frink
bart_simpson		  lionel_hutz	       rainier_wolfcastle
carl_carlson		  lisa_simpson	       ralph_wiggum
charles_montgomery_burns  maggie_simpson       selma_bouvier
chief_wiggum		  marge_simpson        sideshow_bob
cletus_spuckler		  martin_prince        sideshow_mel
comic_book_guy		  mayor_quimby	       simpsons_dataset
disco_stu		  milhouse_van_houten  snake_jailbird
edna_krabappel		  miss_hoover	       troy_mcclure
fat_tony		  moe_szyslak	       waylon_smithers
gil			  ned_flanders
groundskeeper_willie	  nelson_muntz


In [4]:
dataset_dir = target_path
conteo_imagenes = {}

for cls in sorted(os.listdir(dataset_dir)):
    cls_path = os.path.join(dataset_dir, cls)
    if os.path.isdir(cls_path):
        imagenes = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]
        conteo_imagenes[cls] = len(imagenes)

## Mostrar resultados ordenados
for cls, count in sorted(conteo_imagenes.items(), key=lambda x: x[1]):
    print(f"{cls:30s} -> {count} imágenes")

simpsons_dataset               -> 0 imágenes
lionel_hutz                    -> 3 imágenes
disco_stu                      -> 8 imágenes
troy_mcclure                   -> 8 imágenes
miss_hoover                    -> 17 imágenes
fat_tony                       -> 27 imágenes
gil                            -> 27 imágenes
otto_mann                      -> 32 imágenes
sideshow_mel                   -> 40 imágenes
agnes_skinner                  -> 42 imágenes
rainier_wolfcastle             -> 45 imágenes
cletus_spuckler                -> 47 imágenes
snake_jailbird                 -> 55 imágenes
professor_john_frink           -> 65 imágenes
martin_prince                  -> 71 imágenes
patty_bouvier                  -> 72 imágenes
ralph_wiggum                   -> 89 imágenes
carl_carlson                   -> 98 imágenes
selma_bouvier                  -> 103 imágenes
barney_gumble                  -> 106 imágenes
groundskeeper_willie           -> 121 imágenes
maggie_simpson                 -> 1

Como Lionel solo tiene tres imágenes, lo quitamos del entrenamiento

In [5]:
!rm -r ./simpsons_dataset/lionel_hutz

Ahora generaremos diferentes carpetas separando los datos de train, test y validación

In [6]:
## Definimos los paths
ORIGINAL_DATASET_DIR = './simpsons_dataset'
BASE_OUTPUT_DIR = './simpsons_split_dataset'

## Porcentajes; he optado por generar sets de train, test y validación.
## Pero podéis variar los porcentajes
train_pct = 0.7
val_pct = 0.15
test_pct = 0.15

## Crear estructura de carpetas
splits = ['train', 'val', 'test']
classes = os.listdir(ORIGINAL_DATASET_DIR)
classes = [cls for cls in classes if os.path.isdir(os.path.join(ORIGINAL_DATASET_DIR, cls))]

for split in splits:
    for cls in classes:
        os.makedirs(os.path.join(BASE_OUTPUT_DIR, split, cls), exist_ok=True)

## Dividir y copiar imágenes
for cls in classes:
    cls_path = os.path.join(ORIGINAL_DATASET_DIR, cls)
    images = [f for f in os.listdir(cls_path) if f.endswith('.jpg')]
    random.shuffle(images)

    # Asignación predeterminada
    train, val, test = [], [], []

    if len(images) >= 3:
        train, temp = train_test_split(images, train_size=train_pct, random_state=42)
        val, test = train_test_split(temp, test_size=test_pct / (test_pct + val_pct), random_state=42)
    else:
        # Si hay muy pocas imágenes, lo mandamos todo al entrenamiento
        train = images
        print(f"[Aviso] Clase '{cls}' tiene muy pocas imágenes ({len(images)}). Se asignan todas a entrenamiento.")

    for img_list, split in zip([train, val, test], ['train', 'val', 'test']):
        for img in img_list:
            src = os.path.join(cls_path, img)
            dst = os.path.join(BASE_OUTPUT_DIR, split, cls, img)
            shutil.copyfile(src, dst)

[Aviso] Clase 'simpsons_dataset' tiene muy pocas imágenes (0). Se asignan todas a entrenamiento.


In [7]:
## Crear diccionario de mapeo automáticamente
MAP_CHARACTERS = {i: cls for i, cls in enumerate(sorted(classes))}
MAP_CHARACTERS

{0: 'abraham_grampa_simpson',
 1: 'agnes_skinner',
 2: 'apu_nahasapeemapetilon',
 3: 'barney_gumble',
 4: 'bart_simpson',
 5: 'carl_carlson',
 6: 'charles_montgomery_burns',
 7: 'chief_wiggum',
 8: 'cletus_spuckler',
 9: 'comic_book_guy',
 10: 'disco_stu',
 11: 'edna_krabappel',
 12: 'fat_tony',
 13: 'gil',
 14: 'groundskeeper_willie',
 15: 'homer_simpson',
 16: 'kent_brockman',
 17: 'krusty_the_clown',
 18: 'lenny_leonard',
 19: 'lisa_simpson',
 20: 'maggie_simpson',
 21: 'marge_simpson',
 22: 'martin_prince',
 23: 'mayor_quimby',
 24: 'milhouse_van_houten',
 25: 'miss_hoover',
 26: 'moe_szyslak',
 27: 'ned_flanders',
 28: 'nelson_muntz',
 29: 'otto_mann',
 30: 'patty_bouvier',
 31: 'principal_skinner',
 32: 'professor_john_frink',
 33: 'rainier_wolfcastle',
 34: 'ralph_wiggum',
 35: 'selma_bouvier',
 36: 'sideshow_bob',
 37: 'sideshow_mel',
 38: 'simpsons_dataset',
 39: 'snake_jailbird',
 40: 'troy_mcclure',
 41: 'waylon_smithers'}

## Ejercicio

Utilizando Convolutional Neural Networks, entrenar al menos dos clasificadores que sean capaz de reconocer personajes en imágenes de los Simpsons con una accuracy en el dataset de test de, al menos, **90%**. Redactar un informe analizando varias de las alternativas probadas y los resultados obtenidos.

A continuación se detallan una serie de aspectos orientativos que podrían ser analizados en vuestro informe (no es necesario tratar todos ellos, pero cuánta más información podáis aportar mejor a la hora de desarrollar vuestro modelo):

*   Análisis de los datos a utilizar. ¿Qué distribución siguen? ¿Están las clases balanceadas?
*   Análisis de resultados, obtención de métricas de *precision* y *recall* por clase y análisis de qué clases obtienen mejores o peores resultados.
*   Análisis visual de los errores de la red. ¿Qué tipo de imágenes o qué personajes dan más problemas a nuestro modelo?
*   Comparación de modelos CNNs con un modelo de Fully Connected (sin convolución) para este problema.
*   Utilización de distintas arquitecturas CNNs, comentando aspectos como su profundidad, hiperparámetros utilizados, optimizador, uso de técnicas de regularización, *batch normalization*, etc.
*   Utilización de *data augmentation*. Esto puede conseguirse con la clase [ImageDataGenerator](https://keras.io/preprocessing/image/#imagedatagenerator-class) de Keras.


Notas:
* Los datos están en una única carpeta, por lo que tendrás que hacer el split entre train y test
* Recuerda partir los datos en training/validation para tener una buena estimación de los valores que nuestro modelo tendrá en los datos de test, así como comprobar que no estamos cayendo en overfitting. Una posible partición puede ser 80 / 20.
* No es necesario mostrar en el notebook las trazas de entrenamiento de todos los modelos entrenados, si bien una buena idea seria guardar gráficas de esos entrenamientos para el análisis. Sin embargo, **se debe mostrar el entrenamiento completo de al menos los dos mejores modelos obtenidos y la evaluación de los datos de test con estos modelos**.
* Las imágenes **no están normalizadas**. Hay que normalizarlas como hemos hecho en trabajos anteriores.
* El test set del problema tiene imágenes un poco más sencillas de identificar, por lo que es posible encontrarse con métricas en el test set bastante mejores que en el training set.